# One-vs-Rest Classification Analysis

---
## 1. Setup & Data Loading

In [1]:
# =============================================================================
# LIBRARY IMPORTS
# =============================================================================

# Core Libraries
import numpy as np
import pandas as pd
import json
from pathlib import Path
import warnings
from collections import Counter
import re

# Machine Learning Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, brier_score_loss, cohen_kappa_score, confusion_matrix
)

# Statistical Analysis
from scipy import stats
from scipy.stats import (
    wilcoxon, ttest_rel, mannwhitneyu, kruskal, 
    fisher_exact, chi2_contingency, binom
)
from statsmodels.stats.multitest import multipletests

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Configuration
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
sns.set_context('notebook', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

# Color Palette
COLORS = {
    'ovr': '#e6550d',        # Orange
    'multinomial': '#2b8cbe', # Blue
    'rest': '#31a354',        # Green
    'task': '#de2d26',        # Red
    'positive': '#b2182b',
    'negative': '#2166ac',
    'neutral': '#f7f7f7'
}

print("✓ Libraries imported successfully")
print(f"  NumPy: {np.__version__}")
print(f"  Pandas: {pd.__version__}")

✓ Libraries imported successfully
  NumPy: 2.3.4
  Pandas: 2.3.3


In [2]:
# =============================================================================
# PATH CONFIGURATION
# =============================================================================

PROJECT_ROOT = Path('/home/sjoon/projects/brain_connectivity_classifier')
RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'

# Full Model Paths (232 regions)
PATHS = {
    'full_ovr': RESULTS_DIR / 'full_connectivity_analysis' / 'one_vs_rest',
    'full_ovr_task': RESULTS_DIR / 'full_connectivity_analysis' / 'task_testing_one_vs_rest',
    # Left Hemisphere
    'left_ovr': RESULTS_DIR / 'hemisphere_analysis' / 'left_hemisphere' / 'one_vs_rest',
    'left_ovr_task': RESULTS_DIR / 'hemisphere_analysis' / 'left_hemisphere' / 'task_testing_one_vs_rest',
    # Right Hemisphere
    'right_ovr': RESULTS_DIR / 'hemisphere_analysis' / 'right_hemisphere' / 'one_vs_rest',
    'right_ovr_task': RESULTS_DIR / 'hemisphere_analysis' / 'right_hemisphere' / 'task_testing_one_vs_rest',
}

# Verify paths
print("Path Verification:")
for name, path in PATHS.items():
    status = "✓" if path.exists() else "✗"
    print(f"  {status} {name}")

Path Verification:
  ✓ full_ovr
  ✓ full_ovr_task
  ✓ left_ovr
  ✓ left_ovr_task
  ✓ right_ovr
  ✓ right_ovr_task


In [3]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def load_json(fp):
    """Load JSON file."""
    with open(fp, 'r') as f:
        return json.load(f)

def load_npy(fp):
    """Load numpy file."""
    return np.load(fp, allow_pickle=True)

def load_csv(fp):
    """Load CSV file."""
    return pd.read_csv(fp)

def safe_divide(num, denom, default=0):
    """Safe division with zero handling."""
    return num / denom if denom > 0 else default

class DataLoader:
    """Structured data loader for model results."""
    
    def __init__(self, base_path, is_task=False):
        self.base_path = Path(base_path)
        self.is_task = is_task
        
    def load_cv(self):
        """Load cross-validation results."""
        return {
            'predictions': load_npy(self.base_path / 'cv_predictions.npy'),
            'probabilities': load_npy(self.base_path / 'cv_probabilities.npy'),
            'true_labels': load_npy(self.base_path / 'cv_true_labels.npy'),
            'confusion_matrix': load_npy(self.base_path / 'confusion_matrix.npy'),
            'metrics': load_json(self.base_path / 'overall_metrics.json')
        }
    
    def load_task(self):
        """Load task testing results."""
        return {
            'predictions': load_npy(self.base_path / 'task_predictions.npy'),
            'probabilities': load_npy(self.base_path / 'task_probabilities.npy'),
            'true_labels': load_npy(self.base_path / 'task_true_labels.npy'),
            'confusion_matrix': load_npy(self.base_path / 'task_confusion_matrix.npy'),
            'summary': load_json(self.base_path / 'task_testing_summary.json')
        }

print("✓ Helper functions defined")

✓ Helper functions defined


In [4]:
# =============================================================================
# DATA LOADING
# =============================================================================

print("="*80)
print("LOADING ALL MODEL DATA")
print("="*80)

# Full Model (232 regions)
print("\n[Full Model - 232 regions]")
full_ovr_cv = DataLoader(PATHS['full_ovr']).load_cv()
full_ovr_task = DataLoader(PATHS['full_ovr_task']).load_task()
print(f"  OvR CV: {len(full_ovr_cv['predictions']):,} samples")
print(f"  OvR Task: {len(full_ovr_task['predictions']):,} samples")

# Left Hemisphere (116 regions)
print("\n[Left Hemisphere - 116 regions]")
left_ovr_cv = DataLoader(PATHS['left_ovr']).load_cv()
left_ovr_task = DataLoader(PATHS['left_ovr_task']).load_task()
print(f"  OvR CV: {len(left_ovr_cv['predictions']):,} samples")

# Right Hemisphere (116 regions)
print("\n[Right Hemisphere - 116 regions]")
right_ovr_cv = DataLoader(PATHS['right_ovr']).load_cv()
right_ovr_task = DataLoader(PATHS['right_ovr_task']).load_task()
print(f"  OvR CV: {len(right_ovr_cv['predictions']):,} samples")

# Region Information
region_info = load_csv('/home/sjoon/projects/brain_connectivity_classifier/data/FULL_region_info.csv')
print(f"\n✓ Loaded region info: {len(region_info)} regions")

LOADING ALL MODEL DATA

[Full Model - 232 regions]
  OvR CV: 51,968 samples
  OvR Task: 46,400 samples

[Left Hemisphere - 116 regions]
  OvR CV: 25,984 samples

[Right Hemisphere - 116 regions]
  OvR CV: 25,984 samples

✓ Loaded region info: 232 regions


#### Accuracy Difference CV - Task Testing

In [5]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

# ============================================================================
# CONFIGURATION
# ============================================================================

BASE_DIR = Path('/home/sjoon/projects/brain_connectivity_classifier/data/results')

PATHS = {
    # Full model (232 regions)
    'full_cv': BASE_DIR / 'full_connectivity_analysis' / 'one_vs_rest' / 'cv_summary.json',
    'full_task': BASE_DIR / 'full_connectivity_analysis' / 'task_testing_one_vs_rest' / 'task_testing_summary.json',
    
    # Left hemisphere (116 regions)
    'left_cv': BASE_DIR / 'hemisphere_analysis' / 'left_hemisphere' / 'one_vs_rest' / 'cv_summary.json',
    'left_task': BASE_DIR / 'hemisphere_analysis' / 'left_hemisphere' / 'task_testing_one_vs_rest' / 'task_testing_summary.json',
    
    # Right hemisphere (116 regions)
    'right_cv': BASE_DIR / 'hemisphere_analysis' / 'right_hemisphere' / 'one_vs_rest' / 'cv_summary.json',
    'right_task': BASE_DIR / 'hemisphere_analysis' / 'right_hemisphere' / 'task_testing_one_vs_rest' / 'task_testing_summary.json',
}

# ============================================================================
# LOAD AND EXTRACT METRICS
# ============================================================================
results = []

for model_name, cv_path, task_path in [
    ('Full', PATHS['full_cv'], PATHS['full_task']),
    ('Left', PATHS['left_cv'], PATHS['left_task']),
    ('Right', PATHS['right_cv'], PATHS['right_task'])
]:    
    # Load CV summary
    with open(cv_path, 'r') as f:
        cv_data = json.load(f)
    
    # Load Task summary
    with open(task_path, 'r') as f:
        task_data = json.load(f)
    
    # Extract fold-wise accuracies
    fold_metrics = cv_data['fold_metrics']
    train_accs = [fold['train_accuracy'] for fold in fold_metrics]
    val_accs = [fold['val_accuracy'] for fold in fold_metrics]
    
    # Calculate metrics (convert to percentages)
    cv_train_mean = np.mean(train_accs) * 100
    cv_val_mean = np.mean(val_accs) * 100
    generalization_gap = cv_train_mean - cv_val_mean
    task_acc = task_data['task_test_accuracy'] * 100
    accuracy_drop = cv_val_mean - task_acc
    
    results.append({
        'Model': model_name,
        'CV Train Acc (%)': f"{cv_train_mean:.2f}%",
        'CV Val Acc (%)': f"{cv_val_mean:.2f}%",
        'Generalization Gap (%)': f"{generalization_gap:.2f}%",
        'Task Acc (%)': f"{task_acc:.2f}%",
        'Accuracy Drop (%)': f"{accuracy_drop:.2f}%"
    })

# Create and display table
df = pd.DataFrame(results)

print("\n" + "=" * 80)
print("HEMISPHERE COMPARISON - ACCURACY METRICS (PERCENTAGES)")
print("=" * 80)
print(df.to_string(index=False))



HEMISPHERE COMPARISON - ACCURACY METRICS (PERCENTAGES)
Model CV Train Acc (%) CV Val Acc (%) Generalization Gap (%) Task Acc (%) Accuracy Drop (%)
 Full           82.20%         76.24%                  5.96%       73.04%             3.21%
 Left           95.70%         92.79%                  2.91%       88.75%             4.04%
Right           95.44%         92.48%                  2.96%       87.97%             4.51%


In [6]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# ============================================================================
# DATA PREPARATION
# ============================================================================

cols_to_plot = ['CV Val Acc (%)', 'Task Acc (%)']

for col in cols_to_plot:
    df[col] = pd.to_numeric(df[col].astype(str).str.rstrip('%'), errors='coerce')

models = df['Model']
val_acc = df['CV Val Acc (%)']
test_acc = df['Task Acc (%)']

# ============================================================================
# PROFESSIONAL PLOTLY VISUALIZATION
# ============================================================================

fig = go.Figure()

# Add Validation Accuracy bars
fig.add_trace(go.Bar(
    name='Validation Accuracy',
    x=models,
    y=val_acc,
    text=[f'{val:.1f}%' for val in val_acc],
    textposition='outside',
    marker=dict(
        color='#4C72B0',
        line=dict(color='#2E5078', width=1.5)
    ),
    hovertemplate='<b>%{x}</b><br>Validation Accuracy: %{y:.1f}%<extra></extra>'
))

# Add Test Accuracy bars
fig.add_trace(go.Bar(
    name='Test Accuracy',
    x=models,
    y=test_acc,
    text=[f'{val:.1f}%' for val in test_acc],
    textposition='outside',
    marker=dict(
        color='#55A868',
        line=dict(color='#3A7047', width=1.5)
    ),
    hovertemplate='<b>%{x}</b><br>Test Accuracy: %{y:.1f}%<extra></extra>'
))

# Update layout for professional appearance
fig.update_layout(
    title=dict(
        text='Model Performance One-vs-Rest: Validation vs Test Accuracy',
        font=dict(size=20, family='Arial, sans-serif', color='#2c3e50'),
        x=0.5,
        xanchor='center'
    ),
    xaxis=dict(
        title=dict(text='Model', font=dict(size=14, color='#2c3e50')),
        tickfont=dict(size=12, color='#2c3e50'),
        showgrid=False,
        showline=True,
        linewidth=2,
        linecolor='#34495e'
    ),
    yaxis=dict(
        title=dict(text='Accuracy (%)', font=dict(size=14, color='#2c3e50')),
        tickfont=dict(size=12, color='#2c3e50'),
        showgrid=True,
        gridcolor='#ecf0f1',
        gridwidth=1,
        showline=True,
        linewidth=2,
        linecolor='#34495e',
        range=[0, max(val_acc.max(), test_acc.max()) * 1.15]
    ),
    barmode='group',
    bargap=0.2,
    bargroupgap=0.1,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial, sans-serif'),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1,
        font=dict(size=12, color='#2c3e50'),
        bgcolor='rgba(255, 255, 255, 0.8)',
        bordercolor='#bdc3c7',
        borderwidth=1
    ),
    hoverlabel=dict(
        bgcolor='white',
        font_size=12,
        font_family='Arial, sans-serif'
    ),
    margin=dict(t=100, b=80, l=80, r=40),
    height=500,
    width=900
)

fig.show()


In [7]:
# Set base directory for all data files
base_path = '/home/sjoon/projects/brain_connectivity_classifier/data'

# Load region metadata for different brain parcellations
full_region_info = pd.read_csv(f'{base_path}/FULL_region_info.csv')
lh_region_info = pd.read_csv(f'{base_path}/LH_region_info.csv')
rh_region_info = pd.read_csv(f'{base_path}/RH_region_info.csv')

# Define analysis configurations for full brain and hemisphere-specific analyses
analysis_configs = {
    'Full Connectivity': {
        'pred_path': f'{base_path}/results/full_connectivity_analysis/task_testing_one_vs_rest/task_predictions.npy',
        'true_path': f'{base_path}/results/full_connectivity_analysis/task_testing_one_vs_rest/task_true_labels.npy',
        'output_dir': f'{base_path}/results/full_connectivity_analysis/task_testing_one_vs_rest/',
        'region_filter': 'full'
    },
    'Left Hemisphere': {
        'pred_path': f'{base_path}/results/hemisphere_analysis/left_hemisphere/task_testing_one_vs_rest/task_predictions.npy',
        'true_path': f'{base_path}/results/hemisphere_analysis/left_hemisphere/task_testing_one_vs_rest/task_true_labels.npy',
        'output_dir': f'{base_path}/results/hemisphere_analysis/left_hemisphere/task_testing_one_vs_rest/',
        'region_filter': 'left'
    },
    'Right Hemisphere': {
        'pred_path': f'{base_path}/results/hemisphere_analysis/right_hemisphere/task_testing_one_vs_rest/task_predictions.npy',
        'true_path': f'{base_path}/results/hemisphere_analysis/right_hemisphere/task_testing_one_vs_rest/task_true_labels.npy',
        'output_dir': f'{base_path}/results/hemisphere_analysis/right_hemisphere/task_testing_one_vs_rest/',
        'region_filter': 'right'
    }
}

# Map region filters to their corresponding metadata DataFrames
region_sets = {
    'left': lh_region_info,
    'right': rh_region_info,
    'full': full_region_info
}

print("✓ Analysis configurations defined")

def get_comprehensive_report(analysis_configs, region_sets, validation_df):
    """
    Generate comprehensive performance and error analysis report.
    
    Parameters:
    - analysis_configs: Dictionary of analysis configurations
    - region_sets: Dictionary mapping region filters to metadata DataFrames
    - validation_df: DataFrame containing validation accuracy data
    
    Returns:
    - DataFrame with performance metrics and error breakdowns
    """
    # Extract validation accuracies from source DataFrame
    val_series = validation_df.set_index('Model')['CV Val Acc (%)']
    if val_series.dtype == 'object':
        val_series = val_series.str.replace('%', '')
    val_map = pd.to_numeric(val_series, errors='coerce').to_dict()
    
    results = []
    
    for name, config in analysis_configs.items():
        # Get validation accuracy for this analysis
        val_acc = next((v for k, v in val_map.items() if k in name or name in k), 0)
        
        # Load predictions and true labels
        preds = np.load(config['pred_path'], allow_pickle=True)
        labels = np.load(config['true_path'], allow_pickle=True)
        
        # Get appropriate region metadata
        meta = region_sets[config['region_filter']].set_index('region_idx')
        
        # Calculate test accuracy
        test_acc = np.mean(preds == labels) * 100
        
        # Analyze misclassifications
        errors = pd.DataFrame({'t': labels, 'p': preds})
        errors = errors[errors.t != errors.p]  # Keep only errors
        
        # Join with metadata for true and predicted regions
        errors = errors.join(meta[['hemisphere', 'network']].add_prefix('t_'), on='t')
        errors = errors.join(meta[['hemisphere', 'network']].add_prefix('p_'), on='p')
        
        # Categorize errors by hemisphere and network agreement
        same_hemisphere = errors.t_hemisphere == errors.p_hemisphere
        same_network = errors.t_network == errors.p_network
        total_errors = len(errors)
        
        # Format error counts with percentages
        def format_count(mask):
            if total_errors > 0:
                count = len(errors[mask])
                pct = (count / total_errors) * 100
                return f"{count} ({pct:.1f}%)"
            return "0 (0%)"
        
        # Compile results for this analysis
        results.append(pd.Series({
            ('Performance', 'Val Acc (%)'): f"{val_acc:.2f}",
            ('Performance', 'Test Acc (%)'): f"{test_acc:.2f}",
            ('Performance', 'Drop'): f"{val_acc - test_acc:.2f}",
            ('Breakdown', 'Total Errs'): total_errors,
            ('Breakdown', 'Same H/Same N'): format_count(same_hemisphere & same_network),
            ('Breakdown', 'Same H/Diff N'): format_count(same_hemisphere & ~same_network),
            ('Breakdown', 'Diff H/Same N'): format_count(~same_hemisphere & same_network),
            ('Breakdown', 'Diff H/Diff N'): format_count(~same_hemisphere & ~same_network)
        }, name=name))
    
    return pd.DataFrame(results)

# Generate and display comprehensive report
final_report = get_comprehensive_report(analysis_configs, region_sets, df)
display(final_report)


✓ Analysis configurations defined


Performance                     Breakdown                \
                  Val Acc (%) Test Acc (%)  Drop Total Errs Same H/Same N   
Full Connectivity       76.24        73.04  3.20      12511  2056 (16.4%)   
Left Hemisphere         92.79        88.75  4.04       2611    168 (6.4%)   
Right Hemisphere        92.48        87.97  4.51       2791    163 (5.8%)   

                                                             
                  Same H/Diff N Diff H/Same N Diff H/Diff N  
Full Connectivity  5001 (40.0%)  2613 (20.9%)  2841 (22.7%)  
Left Hemisphere    2443 (93.6%)      0 (0.0%)      0 (0.0%)  
Right Hemisphere   2628 (94.2%)      0 (0.0%)      0 (0.0%)

In [8]:
# --- 1. Data Preprocessing (Consolidated) ---
df_breakdown = final_report['Breakdown'].copy()
error_cats = ['Same H/Same N', 'Same H/Diff N', 'Diff H/Same N', 'Diff H/Diff N']

def parse_metrics(val_str):
    val_str = str(val_str)
    count = int(re.search(r'^\d+', val_str).group()) if re.search(r'^\d+', val_str) else 0
    pct = float(re.search(r'\((.*?)%\)', val_str).group(1)) if re.search(r'\(', val_str) else 0.0
    return count, pct

plot_data = []
for model_name in df_breakdown.index:
    for cat in error_cats:
        count, pct = parse_metrics(df_breakdown.loc[model_name, cat])
        plot_data.append({'Model': model_name, 'Category': cat, 'Count': count, 'Pct': pct})

df_plot = pd.DataFrame(plot_data)

# --- 2. Build Compact Plot ---
colors = {'Same H/Same N': '#1A5276', 'Same H/Diff N': '#1D8348', 
          'Diff H/Same N': '#7D3C98', 'Diff H/Diff N': '#2E4053'}

fig = go.Figure()

for cat in error_cats:
    sub = df_plot[df_plot['Category'] == cat]
    labels = [f"n={c}<br>{p:.1f}%" if p > 8.0 else "" for c, p in zip(sub['Count'], sub['Pct'])]
    
    fig.add_trace(go.Bar(
        name=cat, x=sub['Model'], y=sub['Pct'], text=labels,
        textposition='inside', insidetextanchor='middle',
        textfont=dict(size=11, family="Arial", color="white"),
        marker=dict(color=colors[cat], line=dict(color='white', width=1)),
        customdata=sub['Count'],
        hovertemplate="<b>%{x}</b><br>%{customdata} errors (%{y:.1f}%)<extra></extra>"
    ))

# --- 3. Layout Configuration (Compact Sizes) ---
fig.update_layout(
    title=dict(text='<b>Error Composition by Model</b>', font=dict(size=16)),
    barmode='stack',
    template='plotly_white',
    height=450, width=700, # Reduced size
    margin=dict(l=50, r=50, t=60, b=100), # Tight margins
    legend=dict(
        orientation="h", yanchor="top", y=-0.2, 
        xanchor="center", x=0.5,
        font=dict(size=10)
    ),
    yaxis=dict(title='Errors (%)', range=[0, 100], tickfont=dict(size=10)),
    xaxis=dict(tickfont=dict(size=11))
)

fig.show()

In [9]:
# 1. Extract values from your final_report
full_errors = int(final_report.loc['Full Connectivity', ('Breakdown', 'Total Errs')])
lh_errors = int(final_report.loc['Left Hemisphere', ('Breakdown', 'Total Errs')])
rh_errors = int(final_report.loc['Right Hemisphere', ('Breakdown', 'Total Errs')])

# 2. Calculations
sum_hemi_errors = lh_errors + rh_errors
error_reduction = full_errors - sum_hemi_errors
reduction_percentage = (error_reduction / full_errors) * 100

# 3. Output Results
print(f"Total Errors (Full Connectivity): {full_errors:,} \n")
print(f"Total Errors (Left Hemisphere):   {lh_errors:,}")
print(f"Total Errors (Right Hemisphere):  {rh_errors:,}")
print(f"Total Errors (LH + RH Combined):  {sum_hemi_errors:,}\n")
print(f"Absolute Error Reduction:         {error_reduction:,}")
print(f"Percentage Error Reduction:       {reduction_percentage:.2f}%")

Total Errors (Full Connectivity): 12,511 

Total Errors (Left Hemisphere):   2,611
Total Errors (Right Hemisphere):  2,791
Total Errors (LH + RH Combined):  5,402

Absolute Error Reduction:         7,109
Percentage Error Reduction:       56.82%


In [10]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix
import numpy as np

# 1. Create a consolidated name mapping from your region info dataframes
# This ensures that even if a region is in LH or RH, it maps correctly
name_map = {
    **lh_region_info.set_index('region_idx')['region_name'].to_dict(),
    **rh_region_info.set_index('region_idx')['region_name'].to_dict()
}

# Generate the full list of names for all 116 indices (1-116)
region_indices = np.arange(1, 117)
region_names = [name_map.get(idx, f"Region {idx}") for idx in region_indices]

def get_error_cm(model_name):
    config = analysis_configs[model_name]
    preds = np.load(config['pred_path'])
    trues = np.load(config['true_path'])
    
    # Calculate confusion matrix (116x116)
    cm = confusion_matrix(trues, preds, labels=region_indices)
    
    # Set diagonal to zero to hide correct classifications
    np.fill_diagonal(cm, 0)
    return cm

# 2. Extract matrices
cm_lh_err = get_error_cm('Left Hemisphere')
cm_rh_err = get_error_cm('Right Hemisphere')

# 3. Create interactive Subplots
fig = make_subplots(
    rows=1, cols=2, 
    subplot_titles=("<b>Left Hemisphere Error Patterns</b>", "<b>Right Hemisphere Error Patterns</b>"),
    horizontal_spacing=0.1
)

# Shared Heatmap Configuration
# Using names for x and y ensures they appear in the hover text
heatmap_args = dict(
    x=region_names,
    y=region_names,
    colorscale='Viridis',
    hovertemplate="<b>True: %{y}</b><br>Pred: %{x}<br>Count: %{z}<extra></extra>"
)

fig.add_trace(go.Heatmap(z=cm_lh_err, showscale=False, **heatmap_args), row=1, col=1)
fig.add_trace(go.Heatmap(z=cm_rh_err, showscale=True, 
                         colorbar=dict(title="Error Count"), **heatmap_args), row=1, col=2)

# 4. Refined Layout
fig.update_layout(
    title=dict(text='<b>Fine-Grained Region Confusion Analysis (Anatomical Labels)</b>', x=0.5),
    template='plotly_white',
    height=650, 
    width=1150,
    margin=dict(l=150, r=80, t=100, b=150) # Increased margins for long region names
)

# To prevent 116 names from overlapping, we only show numeric ticks on the axis 
# but the HOVER will still show the full region name strings.
fig.update_xaxes(title_text="Predicted Region", tickmode='array', 
                 tickvals=region_names[::10], ticktext=np.arange(1, 117, 10))
fig.update_yaxes(title_text="True Region", tickmode='array', 
                 tickvals=region_names[::10], ticktext=np.arange(1, 117, 10))

fig.show()

In [11]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Load Coordinate Data
path = '/home/sjoon/projects/brain_connectivity_classifier/data/network_files/'
coords_df = pd.read_csv(f'{path}/Schaefer2018_200Parcels_17Networks_order_FSLMNI152_2mm.csv')

# Create a mapping: ROI Label -> (R, A, S)
coords_map = coords_df.set_index('ROI Label')[['R', 'A', 'S']].to_dict('index')

# 2. Define Distance Threshold (in mm)
dist_threshold = 20 

# 3. Generate Distance Mask for the 116 regions
num_regions = 116
dist_mask = np.ones((num_regions, num_regions), dtype=bool)

for i in range(num_regions):
    for j in range(num_regions):
        idx_i, idx_j = i + 1, j + 1 # ROI Labels are 1-indexed
        if idx_i in coords_map and idx_j in coords_map:
            p1 = np.array([coords_map[idx_i]['R'], coords_map[idx_i]['A'], coords_map[idx_i]['S']])
            p2 = np.array([coords_map[idx_j]['R'], coords_map[idx_j]['A'], coords_map[idx_j]['S']])
            dist = np.linalg.norm(p1 - p2)
            if dist < dist_threshold:
                dist_mask[i, j] = False
        else:
            # If coordinates are missing for a label, we keep the error (don't filter)
            dist_mask[i, j] = True

# 4. Filter the existing Error Confusion Matrices
cm_lh_distal = cm_lh_err * dist_mask
cm_rh_distal = cm_rh_err * dist_mask

# Convert zeros to NaN for better sparse visualization in Plotly
cm_lh_plot = cm_lh_distal.astype(float)
cm_lh_plot[cm_lh_plot == 0] = np.nan

cm_rh_plot = cm_rh_distal.astype(float)
cm_rh_plot[cm_rh_plot == 0] = np.nan

# 5. Visualize "Hard" Distal Errors Only
fig = make_subplots(
    rows=1, cols=2, 
    subplot_titles=(f"<b>LH: Distal Errors (>{dist_threshold}mm)</b>", 
                    f"<b>RH: Distal Errors (>{dist_threshold}mm)</b>"),
    horizontal_spacing=0.08
)

heatmap_args = dict(
    x=region_names, y=region_names,
    colorscale='Reds', zmin=1, connectgaps=False,
    hovertemplate="<b>True: %{y}</b><br>Pred: %{x}<br>Count: %{z}<extra></extra>"
)

fig.add_trace(go.Heatmap(z=cm_lh_plot, showscale=False, **heatmap_args), row=1, col=1)
fig.add_trace(go.Heatmap(z=cm_rh_plot, showscale=True, 
                         colorbar=dict(title="Error Count"), **heatmap_args), row=1, col=2)

fig.update_layout(
    title=dict(text=f'<b>Analysis of Significant Distal Misclassifications (Threshold: {dist_threshold}mm)</b>', x=0.5),
    template='plotly_white', height=600, width=1100,
    margin=dict(l=150, r=80, t=100, b=150)
)

fig.update_xaxes(title_text="Predicted Region Index", tickvals=region_names[::10], ticktext=np.arange(1, 117, 10))
fig.update_yaxes(title_text="True Region Index", tickvals=region_names[::10], ticktext=np.arange(1, 117, 10))

fig.show()

# --- Summary of Filtered Results ---
lh_removed = np.sum(cm_lh_err) - np.sum(cm_lh_distal)
rh_removed = np.sum(cm_rh_err) - np.sum(cm_rh_distal)

print(f"Nearby errors removed (LH): {int(lh_removed)} ({(lh_removed/np.sum(cm_lh_err)*100):.1f}% of LH errors)")
print(f"Nearby errors removed (RH): {int(rh_removed)} ({(rh_removed/np.sum(cm_rh_err)*100):.1f}% of RH errors)")

FileNotFoundError: [Errno 2] No such file or directory: '/home/sjoon/projects/brain_connectivity_classifier/data/network_files//Schaefer2018_200Parcels_17Networks_order_FSLMNI152_2mm.csv'

In [ ]:
# 1. Map Network Names for all 116 regions
# We assume region_names contains the strings like '17Networks_LH_VisCent_ExStr_1'
def get_network(name):
    try:
        # Extracts 'VisCent', 'SomMot', etc. from the Schaefer name format
        return name.split('_')[2]
    except:
        return "Unknown"

# Create a mask: True if row (True Region) and col (Pred Region) share the same network
net_match_mask = np.zeros((num_regions, num_regions), dtype=bool)
for i in range(num_regions):
    for j in range(num_regions):
        if get_network(region_names[i]) == get_network(region_names[j]):
            net_match_mask[i, j] = True

def analyze_removed_types(cm_full, distal_mask, net_mask, hemi_label):
    # Identify which cells were removed (had errors but were < threshold distance)
    removed_mask = (cm_full > 0) & (~distal_mask)
    
    total_removed = np.sum(cm_full[removed_mask])
    same_net_removed = np.sum(cm_full[removed_mask & net_mask])
    diff_net_removed = np.sum(cm_full[removed_mask & ~net_mask])
    
    print(f"\n--- {hemi_label} Error Reduction Analysis ---")
    print(f"Total Errors Removed: {int(total_removed)}")
    if total_removed > 0:
        print(f"  • Same-Network Errors: {int(same_net_removed)} ({ (same_net_removed/total_removed*100):.1f}%)")
        print(f"  • Different-Network Errors: {int(diff_net_removed)} ({ (diff_net_removed/total_removed*100):.1f}%)")

# Run the analysis
analyze_removed_types(cm_lh_err, dist_mask, net_match_mask, "Left Hemisphere")
analyze_removed_types(cm_rh_err, dist_mask, net_match_mask, "Right Hemisphere")


--- Left Hemisphere Error Reduction Analysis ---
Total Errors Removed: 147
  • Same-Network Errors: 105 (71.4%)
  • Different-Network Errors: 42 (28.6%)

--- Right Hemisphere Error Reduction Analysis ---
Total Errors Removed: 152
  • Same-Network Errors: 106 (69.7%)
  • Different-Network Errors: 46 (30.3%)


In [ ]:
def calculate_filtered_accuracy(model_name, cm_error, distal_mask, original_acc):
    """
    Calculates updated accuracy by treating nearby errors as non-errors.
    """
    # 1. Get total error count and total samples
    # We can back-calculate total samples from original accuracy
    total_errors = np.sum(cm_error)
    # total_samples = total_errors / (1 - (original_acc / 100))
    # Alternatively, more accurately from the config:
    preds = np.load(analysis_configs[model_name]['pred_path'])
    total_samples = len(preds)
    
    # 2. Identify errors to be removed (those where distance < 20mm)
    # These are errors present in cm_error but masked (False) in distal_mask
    nearby_errors_count = np.sum(cm_error[~distal_mask])
    
    # 3. Calculate New Error Count and Filtered Accuracy
    remaining_errors = total_errors - nearby_errors_count
    filtered_accuracy = ((total_samples - remaining_errors) / total_samples) * 100
    
    return total_samples, total_errors, remaining_errors, filtered_accuracy

# --- Execute Calculations ---
results_filtered = []

for name in ["Left Hemisphere", "Right Hemisphere"]:
    # Use the appropriate error matrix
    cm_err = cm_lh_err if "Left" in name else cm_rh_err
    
    # Get original accuracy from your final_report table
    orig_acc = float(final_report.loc[name, ('Performance', 'Test Acc (%)')])
    
    total, old_err, new_err, filt_acc = calculate_filtered_accuracy(
        name, cm_err, dist_mask, orig_acc
    )
    
    results_filtered.append({
        'Model': name,
        'Original Accuracy (%)': f"{orig_acc:.2f}%",
        'Filtered Accuracy (%)': f"{filt_acc:.2f}%",
        'Accuracy Gain (%)': f"+{(filt_acc - orig_acc):.2f}%",
        'Errors Removed': int(old_err - new_err),
        'Remaining (Distal) Errors': int(new_err)
    })

# Display the summary table
df_filtered_acc = pd.DataFrame(results_filtered).set_index('Model')
display(df_filtered_acc)

,Original Accuracy (%),Filtered Accuracy (%),Accuracy Gain (%),Errors Removed,Remaining (Distal) Errors
Model,,,,,
Left Hemisphere,88.78%,89.65%,+0.87%,147,2402
Right Hemisphere,87.92%,88.59%,+0.67%,152,2646


### Recall Vs Precision (232 Regions)

In [ ]:
# =============================================================================
# PER-REGION DETAILED ERROR PROFILE ANALYSIS
# =============================================================================

# 1. Define Yeo-17 networks for accurate cortical/subcortical tagging
yeo17_networks = [
    'VisCent', 'VisPeri', 'SomMotA', 'SomMotB',
    'DorsAttnA', 'DorsAttnB', 'SalVentAttnA', 'SalVentAttnB',
    'LimbicA', 'LimbicB', 
    'ContA', 'ContB', 'ContC',
    'DefaultA', 'DefaultB', 'DefaultC',
    'TempPar'
]

# Create Atlas column (robust, no dependency on previous 'Type')
region_info['Atlas'] = np.where(
    region_info['network'].isin(yeo17_networks),
    'Schaefer17',
    'Tian'
)

# 2. Compute Recall (class-wise accuracy) and Precision
def get_recall(y_t, y_p): 
    return pd.Series(y_p == y_t).groupby(y_t).mean() * 100

def get_precision(y_t, y_p):
    return pd.Series(y_t == y_p).groupby(y_p).mean() * 100

# Recall (aligned to region_info index)
region_info['Rest_Recall'] = get_recall(full_ovr_cv['true_labels'], full_ovr_cv['predictions'])
region_info['Task_Recall'] = get_recall(full_ovr_task['true_labels'], full_ovr_task['predictions'])
region_info['Recall_Gap'] = region_info['Rest_Recall'] - region_info['Task_Recall']

# Precision (reindex handles regions never predicted)
region_info['Rest_Precision'] = get_precision(
    full_ovr_cv['true_labels'], full_ovr_cv['predictions']
).reindex(region_info.index, fill_value=np.nan)

region_info['Task_Precision'] = get_precision(
    full_ovr_task['true_labels'], full_ovr_task['predictions']
).reindex(region_info.index, fill_value=np.nan)

region_info['Precision_Gap'] = region_info['Rest_Precision'] - region_info['Task_Precision']

# Relative drops
region_info['Recall_Rel_Drop_%'] = (
    region_info['Recall_Gap'] / region_info['Rest_Recall'].replace(0, np.nan) * 100
)
region_info['Precision_Rel_Drop_%'] = (
    region_info['Precision_Gap'] / region_info['Rest_Precision'].replace(0, np.nan) * 100
)

# 3. Improved printing function (now uses 'Atlas' instead of missing 'Type')
def print_per_region_drop(
    df,
    sort_by='Recall_Gap',
    title_suffix='',
    top_n=30,
    width=180
):
    sorted_df = df.sort_values(sort_by, ascending=False).copy()
    if top_n is not None:
        sorted_df = sorted_df.head(top_n)
        top_str = f"(Top {top_n}) "
    else:
        top_str = ""
    
    print(f"\n{'='*width}")
    print(f"PER-REGION ERROR PROFILE {top_str}{title_suffix}")
    print(f"Sorted by descending {sort_by}")
    print("Recall drop → increased false negatives (harder to detect in task)")
    print("Precision drop → increased false positives (less specific in task)")
    print(f"{'='*width}")
    
    header = (
        f"{'Region':<40} {'Network':<25} {'Atlas':<10} │ "
        f"{'Rest Rec':>10} {'Task Rec':>10} {'Abs Drop':>11} {'Rel Drop':>11} │ "
        f"{'Rest Prec':>11} {'Task Prec':>11} {'Abs Drop':>11} {'Rel Drop':>11}"
    )
    print(header)
    print('-' * width)
    
    for _, row in sorted_df.iterrows():
        rest_rec = f"{row['Rest_Recall']:10.2f}%"
        task_rec = f"{row['Task_Recall']:10.2f}%"
        gap_rec = f"{row['Recall_Gap']:10.2f}pt" if pd.notna(row['Recall_Gap']) else "    -     "
        rel_rec = f"{row['Recall_Rel_Drop_%']:10.1f}%" if pd.notna(row['Recall_Rel_Drop_%']) else "    -     "
        
        rest_prec = f"{row['Rest_Precision']:11.2f}%" if pd.notna(row['Rest_Precision']) else "     -     "
        task_prec = f"{row['Task_Precision']:11.2f}%" if pd.notna(row['Task_Precision']) else "     -     "
        gap_prec = f"{row['Precision_Gap']:10.2f}pt" if pd.notna(row['Precision_Gap']) else "    -     "
        rel_prec = f"{row['Precision_Rel_Drop_%']:10.1f}%" if pd.notna(row['Precision_Rel_Drop_%']) else "    -     "
        
        print(
            f"{row['region_name']:<40} {row['network']:<25} {row['Atlas']:<10} │ "
            f"{rest_rec} {task_rec} {gap_rec} {rel_rec} │ "
            f"{rest_prec} {task_prec} {gap_prec} {rel_prec}"
        )
    
    return sorted_df

# 4. Execute the tables
print("\n" + "="*100)
print("DETAILED PER-REGION ERROR PROFILE ANALYSIS (TOP 30 WORST)")
print("="*100)

# Primary: largest absolute recall drops
_ = print_per_region_drop(
    region_info,
    sort_by='Recall_Gap',
    title_suffix="LARGEST ABSOLUTE RECALL DROP (MOST FN-INCREASE)",
    top_n=30
)

# # Secondary: largest relative recall drops
# _ = print_per_region_drop(
#     region_info,
#     sort_by='Recall_Rel_Drop_%',
#     title_suffix="LARGEST RELATIVE RECALL DROP (PROPORTIONALLY HARDEST HIT)",
#     top_n=30
# )

# Tertiary: largest precision drops
_ = print_per_region_drop(
    region_info,
    sort_by='Precision_Gap',
    title_suffix="LARGEST ABSOLUTE PRECISION DROP (MOST FP-INCREASE)",
    top_n=30
)


DETAILED PER-REGION ERROR PROFILE ANALYSIS (TOP 30 WORST)

PER-REGION ERROR PROFILE (Top 30) LARGEST ABSOLUTE RECALL DROP (MOST FN-INCREASE)
Sorted by descending Recall_Gap
Recall drop → increased false negatives (harder to detect in task)
Precision drop → increased false positives (less specific in task)
Region                                   Network                   Atlas      │   Rest Rec   Task Rec    Abs Drop    Rel Drop │   Rest Prec   Task Prec    Abs Drop    Rel Drop
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
RH_ContA_PFCl_1                          ContA                     Schaefer17 │      79.91%      59.50%      20.41pt       25.5% │       76.82%       88.81%     -11.98pt      -15.6%
LH_DefaultB_Temp_3                       DefaultB                  Schaefer17 │      79.46%      61.50%      17.96pt       22.6% │       83.57%       87.

In [12]:
# 5. Enhanced Scatter Plot Visualization with Marginal Distributions

def plot_gap_scatter_readable(df):
    # 1. Create the main scatter plot with marginal distributions (box plots)
    fig = px.scatter(
        df,
        x='Recall_Gap',
        y='Precision_Gap',
        color='Atlas',
        symbol='Atlas',  # Added symbol for accessibility/clarity
        hover_name='region_name',
        # Add 'network' to hover data to spot patterns
        hover_data={
            'network': True, 
            'Recall_Gap': ':.2f', 
            'Precision_Gap': ':.2f', 
            'Atlas': False
        },
        color_discrete_map={'Schaefer17': '#1f77b4', 'Tian': '#ff7f0e'},
        title="<b>Performance Shift: Rest → Task</b><br><sup>Positive values indicate performance degradation (Gap > 0)</sup>",
        labels={
            'Recall_Gap': 'Recall Gap (Positive = More False Negatives)', 
            'Precision_Gap': 'Precision Gap (Positive = More False Positives)'
        },
        marginal_x="box", # Shows distribution of Recall Gaps
        marginal_y="box", # Shows distribution of Precision Gaps
        template="plotly_white" # Cleaner background for readability
    )

    # 2. Refine the markers
    fig.update_traces(marker=dict(size=10, opacity=0.7, line=dict(width=1, color='DarkSlateGrey')))

    # 3. Add Quadrant Lines (Zero lines)
    fig.add_vline(x=0, line_dash="solid", line_color="black", line_width=1)
    fig.add_hline(y=0, line_dash="solid", line_color="black", line_width=1)

    # 4. Dynamic Quadrant Annotations
    # We use x_ref="paper" to place text relative to the layout (0 to 1) 
    # rather than data coordinates. This prevents text from disappearing if data range changes.
    
    # Top Right (Both Worse)
    fig.add_annotation(
        xref="x domain", yref="y domain",
        x=0.98, y=0.98, showarrow=False,
        text="<b>Both Degrade</b><br>(FN & FP ↑)",
        font=dict(color="red", size=10), align="right",
        bgcolor="rgba(255,255,255,0.8)"
    )

    # Bottom Left (Both Better)
    fig.add_annotation(
        xref="x domain", yref="y domain",
        x=0.02, y=0.02, showarrow=False,
        text="<b>Both Improve</b><br>(Performance ↑)",
        font=dict(color="green", size=10), align="left",
        bgcolor="rgba(255,255,255,0.8)"
    )

    # Top Left (FN Improve, FP Worsen) - Tradeoff
    fig.add_annotation(
        xref="x domain", yref="y domain",
        x=0.02, y=0.98, showarrow=False,
        text="<b>Precision Drops</b><br>(More False Positives)",
        font=dict(color="gray", size=10), align="left"
    )

    # Bottom Right (FN Worsen, FP Improve) - Tradeoff
    fig.add_annotation(
        xref="x domain", yref="y domain",
        x=0.98, y=0.02, showarrow=False,
        text="<b>Recall Drops</b><br>(More False Negatives)",
        font=dict(color="gray", size=10), align="right"
    )

    # 5. Final Layout Polish
    fig.update_layout(
        height=700, 
        width=900, 
        legend_title="Brain Atlas",
        font=dict(family="Arial", size=12),
        legend=dict(
            yanchor="top", y=0.99,
            xanchor="left", x=0.01,
            bgcolor="rgba(255,255,255,0.9)"
        )
    )
    
    fig.show()

# Run it
plot_gap_scatter_readable(region_info)

ValueError: Value of 'x' is not the name of a column in 'data_frame'. Expected one of ['region_idx', 'region_name', 'network', 'hemisphere'] but received: Recall_Gap

In [ ]:
# Mean recall and precision gaps
mean_recall_gap = region_info['Recall_Gap'].mean()
mean_precision_gap = region_info['Precision_Gap'].mean()
print(f"\nMean Recall Gap (Rest - Task): {mean_recall_gap:.2f} points")
print(f"Mean Precision Gap (Rest - Task): {mean_precision_gap:.2f} points")

# Mean Recall and Precision Gaps by Atlas
mean_gaps_by_atlas = region_info.groupby('Atlas')[['Recall_Gap', 'Precision_Gap']].mean().reset_index()
print("\nMean Recall and Precision Gaps by Atlas:")
print(mean_gaps_by_atlas.to_string(index=False))

# Mean recall and precision gaps by network

n7_mapping = {
    'VisCent': 'Visual',
    'VisPeri': 'Visual',
    'SomMotA': 'Somatomotor',
    'SomMotB': 'Somatomotor',
    'DorsAttnA': 'Dorsal Attention',
    'DorsAttnB': 'Dorsal Attention',
    'SalVentAttnA': 'Salience/Ventral Attention',
    'SalVentAttnB': 'Salience/Ventral Attention',
    'LimbicA': 'Limbic',
    'LimbicB': 'Limbic',
    'ContA': 'Control',
    'ContB': 'Control',
    'ContC': 'Control',
    'DefaultA': 'Default',
    'DefaultB': 'Default',
    'DefaultC': 'Default',
    'TempPar': 'Temporal/Parietal'
}

subcortical_mask = region_info['Atlas'] == 'Tian'
region_info.loc[subcortical_mask, 'n7_network'] = 'Subcortical'
region_info.loc[~subcortical_mask, 'n7_network'] = region_info.loc[~subcortical_mask, 'network'].map(n7_mapping)

mean_gaps_by_n7_network = region_info.groupby('n7_network')[['Recall_Gap', 'Precision_Gap']].mean().reset_index()
print("\nMean Recall and Precision Gaps by 7-Network System:")
print(mean_gaps_by_n7_network.to_string(index=False))


Mean Recall Gap (Rest - Task): 3.21 points
Mean Precision Gap (Rest - Task): 2.68 points

Mean Recall and Precision Gaps by Atlas:
     Atlas  Recall_Gap  Precision_Gap
Schaefer17    2.550982       1.962324
      Tian    7.303013       7.191410

Mean Recall and Precision Gaps by 7-Network System:
                n7_network  Recall_Gap  Precision_Gap
                   Control    2.739382       3.225720
                   Default    1.779440       0.357788
          Dorsal Attention    1.531656       0.952474
                    Limbic    1.691327       1.894170
Salience/Ventral Attention    3.128434       2.882080
               Somatomotor    2.747899       2.309840
               Subcortical    7.303013       7.191410
         Temporal/Parietal    2.294643       1.274619
                    Visual    4.045387       2.136913


### Recall By Network

In [ ]:
# =============================================================================
# 3a. Interactive Box Plot: Recall Gap by Network (separate)
# =============================================================================
def plot_recall_gap_by_network_plotly(df):
    fig = px.box(
        df,
        x='network',
        y='Recall_Gap',
        color='Atlas',
        points='outliers',
        hover_name='region_name',
        hover_data={'network': True, 'Recall_Gap': ':.2f'},
        color_discrete_map={'Schaefer17': '#1f77b4', 'Tian': '#ff7f0e'},
        title='Recall Gap Distribution by Network<br>(positive = more FN / less detectable)',
        labels={'Recall_Gap': 'Gap (percentage points)', 'network': 'Network'}
    )
    
    # Dashed zero line
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.8)
    
    # Fixed Y-axis as requested
    fig.update_yaxes(range=[-20, 20], dtick=5)
    
    fig.update_layout(
        height=500,
        width=1200,
        xaxis=dict(tickangle=90, tickfont_size=10),
        legend_title="Atlas"
    )
    
    fig.show()

plot_recall_gap_by_network_plotly(region_info)


Recall
- Recall is TP/(TP+FN) -> Recall means out of all actual cases (Positive) how many it can find correctly. 
- Recall is important in medical cases where we can have more FP because it is more risky classify a patient FN. we can do another test to confirm the patient health. 

Recall Gap
- Recall Gap (Rest Recal  - Task Recal)
- Positive gap → recall drops in task → more false negatives (FN) → the region becomes harder to detect (less detectable/sensitive).
- Negative gap → recall improves in task

Task conditions make many regions harder to detect (↑FN Positive Gap), especially the subcortical regions 

Observations (subcortial)
- Very high positive medians and large outliers (e.g., Pallidum_post, Hippocampus_post, Putamen, Accumbens).
- Gaps often +10 to +20 pt or more → strong recall drops.
- These small/deep structures become much less detectable during task (likely because task alters their connectivity patterns or adds noise).


Observations (cortical)
- Medians mostly near 0 or slightly positive.
- Smaller spreads and fewer extreme outliers.
- Primary sensory/motor networks (VisCent, VisPeri, SomMot) show almost no change.
- Higher-order networks (ContA/B, Default, SalVentAttn) have modest positive medians → some detectability loss.

The biggest detectability problems in task are concentrated in subcortical regions (hippocampus, pallidum, basal ganglia). Cortical regions are more stable, especially early sensory ones.


### Precision By Network

In [ ]:
# =============================================================================
# 3b. Interactive Box Plot: Precision Gap by Network (separate)
# =============================================================================
def plot_precision_gap_by_network_plotly(df):
    fig = px.box(
        df,
        x='network',
        y='Precision_Gap',
        color='Atlas',
        points='outliers',
        hover_name='region_name',
        hover_data={'network': True, 'Precision_Gap': ':.2f'},
        color_discrete_map={'Schaefer17': '#1f77b4', 'Tian': '#ff7f0e'},
        title='Precision Gap Distribution by Network<br>(positive = more FP / less specific)',
        labels={'Precision_Gap': 'Gap (percentage points)', 'network': 'Network'}
    )
    
    # Dashed zero line
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.8)
    
    # Fixed Y-axis as requested
    fig.update_yaxes(range=[-30, 30],dtick=5)
    
    fig.update_layout(
        height=500,
        width=1200,
        xaxis=dict(tickangle=90, tickfont_size=10),
        legend_title="Atlas"
    )
    
    fig.show()

plot_precision_gap_by_network_plotly(region_info)

**Overall Takeaways**

- **Subcortical vulnerability**: Tian networks show worst recall and precision loss → deep structures lose detectability and specificity during tasks.

- **Cortical resilience**: Visual and somatomotor networks remain stable, with minimal connectivity changes.

- **Higher-order sensitivity**: Control, default, and attention networks show moderate recall drops and variable precision changes → task alters their functional organization.

**Failure Modes**

- **Recall loss** → mainly false negatives (missed connections), worst in subcortical regions.

- **Precision loss** → mainly false positives (over-detections), extreme in control/subcortical networks.

**Network-Specific Findings**

1. **Control Networks (ContA/B/C)**: Complete breakdown—harder to detect AND produce false alarms.

2. **Subcortical Structures**: Extreme instability—wide distributions and outliers indicate unpredictable connectivity changes.

3. **Salience/Ventral Attention**: "Promiscuous" connectivity—detected but with incorrect partners.

4. **Visual Networks**: High stability—tight distributions near zero show consistent connectivity.

**Methodological Insight**

Classifier trained on resting-state data, reveals that task states don't simply amplify or dampen \
resting patterns—they fundamentally reorganize connectivity architecture. Some networks (like visual) \
scale predictably,while others (like control and subcortical) undergo structural transformation. \ 

The fact that some networks lose recall while gaining precision (or vice versa) indicates the model's \
decision boundaries shift differently for each brain region. This isn't a global threshold adjustment—it's \
region-specific reconfiguration that reflects genuine biological differences in how networks respond to cognitive demands.

**Clinical Implications**

- Task-based biomarkers may be more informative for executive/limbic regions.

- Visual networks remain stable and reliable for biomarker applications.

## Phase 3 NETWORK DISRUPTION PROFILES

In [ ]:
# =============================================================================
# PHASE 3: NETWORK DISRUPTION PROFILES
# =============================================================================

print("\n" + "="*80)
print("PHASE 3: NETWORK DISRUPTION PROFILES")
print("="*80)

# =============================================================================
# Step 1: Add Atlas Column (matching your reference code)
# =============================================================================

# Define Yeo-17 networks for accurate cortical/subcortical tagging
yeo17_networks = [
    'VisCent', 'VisPeri', 'SomMotA', 'SomMotB',
    'DorsAttnA', 'DorsAttnB', 'SalVentAttnA', 'SalVentAttnB',
    'LimbicA', 'LimbicB', 
    'ContA', 'ContB', 'ContC',
    'DefaultA', 'DefaultB', 'DefaultC',
    'TempPar'
]

# Create Atlas column (robust, no dependency on previous 'Type')
region_info['Atlas'] = np.where(
    region_info['network'].isin(yeo17_networks),
    'Schaefer17',
    'Tian'
)

print(f"✓ Atlas classification complete:")
print(f"  Schaefer17 (Cortical): {(region_info['Atlas'] == 'Schaefer17').sum()} regions")
print(f"  Tian (Subcortical): {(region_info['Atlas'] == 'Tian').sum()} regions \n")


print(f"shape of region_info: {region_info.shape}")
region_info.head()


PHASE 3: NETWORK DISRUPTION PROFILES
✓ Atlas classification complete:
  Schaefer17 (Cortical): 200 regions
  Tian (Subcortical): 32 regions 

shape of region_info: (232, 14)


,region_idx,region_name,network,hemisphere,Atlas,Rest_Recall,Task_Recall,Recall_Gap,Rest_Precision,Task_Precision,Precision_Gap,Recall_Rel_Drop_%,Precision_Rel_Drop_%,n7_network
0,0,LH_VisCent_ExStr_2,VisCent,left,Schaefer17,78.125000,73.0,5.125000,70.564516,64.035088,6.529428,6.560000,9.253133,Visual
1,1,LH_VisCent_ExStr_1,VisCent,left,Schaefer17,86.607143,82.0,4.607143,91.943128,85.416667,6.526461,5.319588,7.098368,Visual
2,2,LH_VisCent_Striate_1,VisCent,left,Schaefer17,90.625000,87.5,3.125000,87.500000,80.275229,7.224771,3.448276,8.256881,Visual
3,3,LH_VisCent_ExStr_3,VisCent,left,Schaefer17,78.571429,72.0,6.571429,77.192982,73.846154,3.346829,8.363636,4.335664,Visual
4,4,LH_VisCent_ExStr_4,VisCent,left,Schaefer17,85.714286,86.5,-0.785714,84.955752,86.069652,-1.113900,-0.916667,-1.311153,Visual


In [ ]:
# =============================================================================
# Step 2: Compute Recall and Precision (matching your methodology)
# =============================================================================

def get_recall(y_t, y_p): 
    """Compute per-class recall (class-wise accuracy)."""
    return pd.Series(y_p == y_t).groupby(y_t).mean() * 100

def get_precision(y_t, y_p):
    """Compute per-class precision."""
    return pd.Series(y_t == y_p).groupby(y_p).mean() * 100

# Recall (aligned to region_info index)
region_info['Rest_Recall'] = get_recall(full_ovr_cv['true_labels'], full_ovr_cv['predictions'])
region_info['Task_Recall'] = get_recall(full_ovr_task['true_labels'], full_ovr_task['predictions'])
region_info['Recall_Gap'] = region_info['Rest_Recall'] - region_info['Task_Recall']

# Precision (reindex handles regions never predicted)
region_info['Rest_Precision'] = get_precision(
    full_ovr_cv['true_labels'], full_ovr_cv['predictions']
).reindex(region_info.index, fill_value=np.nan)

region_info['Task_Precision'] = get_precision(
    full_ovr_task['true_labels'], full_ovr_task['predictions']
).reindex(region_info.index, fill_value=np.nan)

region_info['Precision_Gap'] = region_info['Rest_Precision'] - region_info['Task_Precision']

# Relative drops
region_info['Recall_Rel_Drop_%'] = (
    region_info['Recall_Gap'] / region_info['Rest_Recall'].replace(0, np.nan) * 100
)
region_info['Precision_Rel_Drop_%'] = (
    region_info['Precision_Gap'] / region_info['Rest_Precision'].replace(0, np.nan) * 100
)

print("\n✓ Computed recall and precision metrics for all regions \n")

print(f"shape of region_info: {region_info.shape}")
region_info.head()

In [ ]:
# Calculate bootstrap threshold for Rest→Task gaps

def estimate_rest_task_noise_threshold(cv_data, task_data, n_permutations=1000):
    """Estimate significance threshold for Rest→Task gaps via bootstrap."""
    
    y_true_cv = cv_data['true_labels']
    y_pred_cv = cv_data['predictions']
    y_true_task = task_data['true_labels']
    y_pred_task = task_data['predictions']
    
    n_cv = len(y_true_cv)
    n_task = len(y_true_task)
    
    bootstrap_gaps = []
    
    for _ in range(n_permutations):
        # Bootstrap resample from each dataset
        idx_cv = np.random.choice(n_cv, n_cv, replace=True)
        idx_task = np.random.choice(n_task, n_task, replace=True)
        
        recall_cv = get_recall(y_true_cv[idx_cv], y_pred_cv[idx_cv])
        recall_task = get_recall(y_true_task[idx_task], y_pred_task[idx_task])
        
        # Gap from resampled data
        gap = (recall_cv - recall_task).abs()
        bootstrap_gaps.append(gap.mean())
    
    # 95th percentile = noise ceiling
    threshold = np.percentile(bootstrap_gaps, 95)
    
    return threshold, bootstrap_gaps

threshold, gaps = estimate_rest_task_noise_threshold(full_ovr_cv, full_ovr_task)
print(f"Bootstrap 95th percentile threshold: {threshold:.2f}pt")
print(f"Mean bootstrap gap: {np.mean(gaps):.2f}pt")
print(f"Std: {np.std(gaps):.2f}pt")

Bootstrap 95th percentile threshold: 6.70pt
Mean bootstrap gap: 6.34pt
Std: 0.22pt


In [ ]:
# =============================================================================
# 3A. NETWORK DISRUPTION PROFILES
# =============================================================================
# use emperically- derived threshold

THRESHOLD, _ = estimate_rest_task_noise_threshold(full_ovr_cv, full_ovr_task)
print(f"\nUsing bootstrap-derived threshold: {THRESHOLD:.2f}pt for disruption classification\n")

def classify_disruption_type(recall_gap, precision_gap, threshold=THRESHOLD):
    """
    Classify regions based on their performance gaps.
    
    Parameters:
    -----------
    recall_gap : float
        Recall gap in percentage points (positive = worse in task)
    precision_gap : float
        Precision gap in percentage points (positive = worse in task)
    threshold : float
        Threshold for considering a change significant (default: 5pt)
    
    Returns:
    --------
    str : Disruption type classification
    """
    # Handle NaN values
    if pd.isna(recall_gap) or pd.isna(precision_gap):
        return 'Insufficient Data'
    
    if recall_gap > threshold and precision_gap <= threshold:
        return 'FN-dominated'
    elif precision_gap > threshold and recall_gap <= threshold:
        return 'FP-dominated'
    elif recall_gap > threshold and precision_gap > threshold:
        return 'Both (FN+FP)'
    elif recall_gap < -threshold and precision_gap < -threshold:
        return 'Improved'
    else:
        return 'Stable'

# Classify each region
region_info['Disruption_Type'] = region_info.apply(
    lambda row: classify_disruption_type(row['Recall_Gap'], row['Precision_Gap'], threshold=5),
    axis=1
)

# Calculate error magnitude (Euclidean distance in gap space)
region_info['Error_Magnitude'] = np.sqrt(
    region_info['Recall_Gap'].fillna(0)**2 + region_info['Precision_Gap'].fillna(0)**2
)

print("\n" + "="*80)
print("3A. NETWORK DISRUPTION PROFILES")
print("="*80)

# =============================================================================
# Network-Level Summary Statistics
# =============================================================================

def calculate_network_disruption_profiles(region_df):
    """Calculate disruption profiles for each network."""
    
    # Group by network
    network_summary = []
    
    for network in sorted(region_df['network'].unique()):
        net_regions = region_df[region_df['network'] == network]
        total_regions = len(net_regions)
        
        # Count disruption types
        disruption_counts = net_regions['Disruption_Type'].value_counts()
        
        # Calculate statistics
        summary = {
            'Network': network,
            'Total_Regions': total_regions,
            'FN_Count': disruption_counts.get('FN-dominated', 0),
            'FP_Count': disruption_counts.get('FP-dominated', 0),
            'Both_Count': disruption_counts.get('Both (FN+FP)', 0),
            'Stable_Count': disruption_counts.get('Stable', 0),
            'Improved_Count': disruption_counts.get('Improved', 0),
            'Insufficient_Count': disruption_counts.get('Insufficient Data', 0),
        }
        
        # Calculate percentages
        summary['FN_Pct'] = (summary['FN_Count'] / total_regions) * 100
        summary['FP_Pct'] = (summary['FP_Count'] / total_regions) * 100
        summary['Both_Pct'] = (summary['Both_Count'] / total_regions) * 100
        summary['Stable_Pct'] = (summary['Stable_Count'] / total_regions) * 100
        summary['Improved_Pct'] = (summary['Improved_Count'] / total_regions) * 100
        
        # Average error magnitude
        summary['Avg_Error_Magnitude'] = net_regions['Error_Magnitude'].mean()
        summary['Avg_Recall_Gap'] = net_regions['Recall_Gap'].mean()
        summary['Avg_Precision_Gap'] = net_regions['Precision_Gap'].mean()
        
        network_summary.append(summary)
    
    network_df = pd.DataFrame(network_summary)
    network_df = network_df.sort_values('Avg_Error_Magnitude', ascending=False)
    
    return network_df

network_profiles = calculate_network_disruption_profiles(region_info)

# =============================================================================
# Display Network Summary Table
# =============================================================================

print("\nNetwork Disruption Summary:")
print("="*140)
print(f"{'Network':<20} {'N':>4} │ {'%FN':>6} {'%FP':>6} {'%Both':>6} {'%Stable':>7} {'%Impr':>6} │ "
      f"{'Avg Error':>10} {'Avg Δ Recall':>12} {'Avg Δ Prec':>11}")
print("-"*140)

for _, row in network_profiles.iterrows():
    print(f"{row['Network']:<20} {row['Total_Regions']:>4} │ "
          f"{row['FN_Pct']:>6.1f} {row['FP_Pct']:>6.1f} {row['Both_Pct']:>6.1f} "
          f"{row['Stable_Pct']:>7.1f} {row['Improved_Pct']:>6.1f} │ "
          f"{row['Avg_Error_Magnitude']:>10.2f} {row['Avg_Recall_Gap']:>12.2f} "
          f"{row['Avg_Precision_Gap']:>11.2f}")

print("="*140)
print(f"Legend: FN = False Negative dominated (harder to detect)")
print(f"        FP = False Positive dominated (less specific)")
print(f"        Both = Both FN and FP increase")
print(f"        Stable = Within ± {THRESHOLD:.2f}pt threshold")
print(f"        Impr = Improved in task state")


Using bootstrap-derived threshold: 6.69pt for disruption classification


3A. NETWORK DISRUPTION PROFILES

Network Disruption Summary:
Network                 N │    %FN    %FP  %Both %Stable  %Impr │  Avg Error Avg Δ Recall  Avg Δ Prec
--------------------------------------------------------------------------------------------------------------------------------------------
Hippocampus_post        2 │    0.0    0.0  100.0     0.0    0.0 │      20.47        15.07       13.81
Pallidum_ant            2 │    0.0    0.0  100.0     0.0    0.0 │      17.36         8.98       14.85
Pallidum_post           2 │    0.0   50.0   50.0     0.0    0.0 │      14.53         8.29       11.05
Amygdala_lat            2 │    0.0   50.0   50.0     0.0    0.0 │      14.25         4.39       12.43
Thalamus_VP             2 │   50.0    0.0   50.0     0.0    0.0 │      13.89        12.33        4.03
Accumbens_shell         2 │    0.0    0.0  100.0     0.0    0.0 │      13.48         9.41        9.65
Hippocamp

In [ ]:
# =============================================================================
# Key Insights by Atlas Type
# =============================================================================

print("\n" + "="*80)
print("DISRUPTION PATTERNS BY ATLAS TYPE")
print("="*80)

for atlas in ['Schaefer17', 'Tian']:
    atlas_regions = region_info[region_info['Atlas'] == atlas]
    
    print(f"\n{atlas} ({len(atlas_regions)} regions):")
    print("-"*80)
    
    disruption_counts = atlas_regions['Disruption_Type'].value_counts()
    
    for dtype, count in disruption_counts.items():
        pct = (count / len(atlas_regions)) * 100
        print(f"  {dtype:<20}: {count:>3} regions ({pct:>5.1f}%)")
    
    print(f"\n  Average Error Magnitude: {atlas_regions['Error_Magnitude'].mean():.2f}")
    print(f"  Average Recall Gap:      {atlas_regions['Recall_Gap'].mean():.2f}pt")
    print(f"  Average Precision Gap:   {atlas_regions['Precision_Gap'].mean():.2f}pt")


DISRUPTION PATTERNS BY ATLAS TYPE

Schaefer17 (200 regions):
--------------------------------------------------------------------------------
  Stable              :  82 regions ( 41.0%)
  FN-dominated        :  51 regions ( 25.5%)
  FP-dominated        :  48 regions ( 24.0%)
  Both (FN+FP)        :  15 regions (  7.5%)
  Improved            :   4 regions (  2.0%)

  Average Error Magnitude: 8.26
  Average Recall Gap:      2.55pt
  Average Precision Gap:   1.96pt

Tian (32 regions):
--------------------------------------------------------------------------------
  Both (FN+FP)        :  16 regions ( 50.0%)
  FN-dominated        :   6 regions ( 18.8%)
  Stable              :   5 regions ( 15.6%)
  FP-dominated        :   5 regions ( 15.6%)

  Average Error Magnitude: 11.61
  Average Recall Gap:      7.30pt
  Average Precision Gap:   7.19pt


In [ ]:
# =============================================================================
# Visualization 1: Network Disruption Stacked Bar Chart
# =============================================================================

def plot_network_disruption_stacked(network_profiles):
    """Create stacked bar chart of disruption types by network."""
    
    fig = go.Figure()
    
    colors = {
        'FN_Pct': '#E74C3C',           # Red - False Negatives
        'FP_Pct': '#F39C12',           # Orange - False Positives
        'Both_Pct': '#9B59B6',         # Purple - Both
        'Stable_Pct': '#2ECC71',       # Green - Stable
        'Improved_Pct': '#3498DB'      # Blue - Improved
    }
    
    labels = {
        'FN_Pct': 'FN-dominated',
        'FP_Pct': 'FP-dominated',
        'Both_Pct': 'Both (FN+FP)',
        'Stable_Pct': 'Stable',
        'Improved_Pct': 'Improved'
    }
    
    for col in ['FN_Pct', 'FP_Pct', 'Both_Pct', 'Stable_Pct', 'Improved_Pct']:
        fig.add_trace(go.Bar(
            name=labels[col],
            x=network_profiles['Network'],
            y=network_profiles[col],
            marker_color=colors[col],
            text=network_profiles[col].round(1),
            textposition='inside',
            texttemplate='%{text:.1f}%',
            hovertemplate='%{x}<br>' + labels[col] + ': %{y:.1f}%<extra></extra>'
        ))
    
    fig.update_layout(
        barmode='stack',
        title={
            'text': 'Network Disruption Profiles: Rest → Task State Transition',
            'x': 0.5,
            'xanchor': 'center'
        },
        xaxis_title='Network',
        yaxis_title='Percentage of Regions (%)',
        height=600,
        width=1200,
        legend_title='Disruption Type',
        legend=dict(
            orientation="v",
            yanchor="top",
            y=0.99,
            xanchor="right",
            x=0.99
        ),
        hovermode='x unified'
    )
    
    fig.update_xaxes(tickangle=45)
    
    return fig

fig_network_disruption = plot_network_disruption_stacked(network_profiles)
fig_network_disruption.show()

In [ ]:
n7_mapping = {
    'VisCent': 'Visual',
    'VisPeri': 'Visual',
    'SomMotA': 'Somatomotor',
    'SomMotB': 'Somatomotor',
    'DorsAttnA': 'Dorsal Attention',
    'DorsAttnB': 'Dorsal Attention',
    'SalVentAttnA': 'Salience/Ventral Attention',
    'SalVentAttnB': 'Salience/Ventral Attention',
    'LimbicA': 'Limbic',
    'LimbicB': 'Limbic',
    'ContA': 'Control',
    'ContB': 'Control',
    'ContC': 'Control',
    'DefaultA': 'Default',
    'DefaultB': 'Default',
    'DefaultC': 'Default',
    'TempPar': 'Temporal/Parietal'
}

# Create a function to map network names to N7 categories
def map_to_n7_category(network_name):
    """Map a network name to its N7 category."""
    # First check if it's in the mapping
    if network_name in n7_mapping:
        return n7_mapping[network_name]
    # Check if any of the mapping keys are in the network name (for partial matches)
    for key in n7_mapping:
        if key in str(network_name):
            return n7_mapping[key]
    # If not found, assume it's subcortical
    return 'Subcortical'

# Apply the mapping to create Category_8 column
network_profiles['Category_8'] = network_profiles['Network'].apply(map_to_n7_category)

# =============================================================================
# Visualization 1: Network Disruption Stacked Bar Chart
# =============================================================================

def plot_network_disruption_stacked(network_df):
    """Create stacked bar chart of disruption types by network."""
    
    # First, we need to aggregate by Category_8 since we want 8 bars
    # Let's create the aggregated data
    category_order = [
        'Visual',
        'Somatomotor', 
        'Dorsal Attention',
        'Salience/Ventral Attention',
        'Limbic',
        'Control',
        'Default',
        'Subcortical'
    ]
    
    # Filter only the categories we want (in case there are others)
    network_df = network_df[network_df['Category_8'].isin(category_order)]
    
    # Create aggregated data by Category_8
    aggregated_data = []
    
    for category in category_order:
        # Get all rows for this category
        cat_rows = network_df[network_df['Category_8'] == category]
        
        if len(cat_rows) == 0:
            continue
            
        # Calculate total regions in this category
        total_regions = cat_rows['Total_Regions'].sum()
        
        # Calculate aggregated percentages (weighted by Total_Regions)
        fn_pct = (cat_rows['FN_Count'].sum() / total_regions) * 100 if total_regions > 0 else 0
        fp_pct = (cat_rows['FP_Count'].sum() / total_regions) * 100 if total_regions > 0 else 0
        both_pct = (cat_rows['Both_Count'].sum() / total_regions) * 100 if total_regions > 0 else 0
        stable_pct = (cat_rows['Stable_Count'].sum() / total_regions) * 100 if total_regions > 0 else 0
        improved_pct = (cat_rows['Improved_Count'].sum() / total_regions) * 100 if total_regions > 0 else 0
        
        aggregated_data.append({
            'Category_8': category,
            'Total_Regions': total_regions,
            'FN_Pct': fn_pct,
            'FP_Pct': fp_pct,
            'Both_Pct': both_pct,
            'Stable_Pct': stable_pct,
            'Improved_Pct': improved_pct
        })
    
    # Create DataFrame from aggregated data
    category_df = pd.DataFrame(aggregated_data)
    
    # Sort by the predefined order
    category_df['Category_8'] = pd.Categorical(category_df['Category_8'], 
                                              categories=category_order, 
                                              ordered=True)
    category_df = category_df.sort_values('Category_8')
        
    # Now create the plot
    fig = go.Figure()
    
    colors = {
        'FN_Pct': '#E74C3C',           # Red - False Negatives
        'FP_Pct': '#F39C12',           # Orange - False Positives
        'Both_Pct': '#9B59B6',         # Purple - Both
        'Stable_Pct': '#2ECC71',       # Green - Stable
        'Improved_Pct': '#3498DB'      # Blue - Improved
    }
    
    labels = {
        'FN_Pct': 'FN-dominated',
        'FP_Pct': 'FP-dominated',
        'Both_Pct': 'Both (FN+FP)',
        'Stable_Pct': 'Stable',
        'Improved_Pct': 'Improved'
    }
    
    for col in ['FN_Pct', 'FP_Pct', 'Both_Pct', 'Stable_Pct', 'Improved_Pct']:
        fig.add_trace(go.Bar(
            name=labels[col],
            x=category_df['Category_8'],
            y=category_df[col],
            marker_color=colors[col],
            text=category_df[col].round(1),
            textposition='inside',
            texttemplate='%{text:.1f}%',
            hovertemplate='%{x}<br>' + labels[col] + ': %{y:.1f}%<br>Total regions: %{customdata}<extra></extra>',
            customdata=category_df['Total_Regions']
        ))
    
    fig.update_layout(
        barmode='stack',
        title={
            'text': 'Network Disruption Profiles: Rest → Task State Transition<br>7 Yeo Networks + Subcortical',
            'x': 0.5,
            'xanchor': 'center',
            'font': dict(size=16)
        },
        xaxis_title='Network Category',
        yaxis_title='Percentage of Regions (%)',
        height=600,
        width=1200,
        legend_title='Disruption Type',
        legend=dict(
            orientation="v",
            yanchor="top",
            y=0.99,
            xanchor="right",
            x=0.99
        ),
        hovermode='x unified'
    )
    
    fig.update_xaxes(tickangle=45)
    fig.update_yaxes(range=[0, 100])
    
    return fig, category_df

# Create and show the plot
fig_network_disruption, category_df = plot_network_disruption_stacked(network_profiles)
fig_network_disruption.show()

In [ ]:
# =============================================================================
# Visualization 3: Error Magnitude Heatmap by Network
# =============================================================================

def plot_error_magnitude_heatmap(region_df):
    """Create heatmap showing average error magnitude by network and disruption type."""
    
    # Pivot table
    pivot_data = region_df.groupby(['network', 'Disruption_Type'])['Error_Magnitude'].mean().reset_index()
    pivot_table = pivot_data.pivot(index='network', columns='Disruption_Type', values='Error_Magnitude')
    
    # Reorder columns
    col_order = ['FN-dominated', 'FP-dominated', 'Both (FN+FP)', 'Stable', 'Improved', 'Insufficient Data']
    pivot_table = pivot_table[[col for col in col_order if col in pivot_table.columns]]
    
    fig = go.Figure(data=go.Heatmap(
        z=pivot_table.values,
        x=pivot_table.columns,
        y=pivot_table.index,
        colorscale='YlOrRd',
        text=pivot_table.values.round(2),
        texttemplate='%{text}',
        textfont={"size": 10},
        colorbar=dict(title="Avg Error<br>Magnitude")
    ))
    
    fig.update_layout(
        title='Average Error Magnitude by Network and Disruption Type',
        xaxis_title='Disruption Type',
        yaxis_title='Network',
        height=800,
        width=900
    )

    return fig

fig_heatmap = plot_error_magnitude_heatmap(region_info)
fig_heatmap.show()

In [ ]:
# =============================================================================
# Visualization 3b: Relative Error Magnitude Heatmap (Normalized by Rest)
# =============================================================================

def plot_relative_error_magnitude_heatmap(region_df):
    """
    Create heatmap showing average relative error magnitude
    normalized by rest performance.
    """

    df = region_df.copy()
    df['Rest_Baseline'] = (df['Rest_Recall'] + df['Rest_Precision']) / 2

    # Avoid division by zero
    df = df[df['Rest_Baseline'] > 0]

    # -------------------------------------------------------------------------
    # Step 2: Normalize error magnitude
    # -------------------------------------------------------------------------
    df['Relative_Error_Magnitude'] = df['Error_Magnitude'] / df['Rest_Baseline']

    # -------------------------------------------------------------------------
    # Step 3: Aggregate by network and disruption type
    # -------------------------------------------------------------------------
    pivot_data = (
        df.groupby(['network', 'Disruption_Type'])['Relative_Error_Magnitude']
          .mean()
          .reset_index()
    )
    pivot_table = pivot_data.pivot(
        index='network',
        columns='Disruption_Type',
        values='Relative_Error_Magnitude'
    )
    # Column order
    col_order = [
        'FN-dominated',
        'FP-dominated',
        'Both (FN+FP)',
        'Stable',
        'Improved',
        'Insufficient Data'
    ]
    pivot_table = pivot_table[[c for c in col_order if c in pivot_table.columns]]

    fig = go.Figure(data=go.Heatmap(
        z=pivot_table.values,
        x=pivot_table.columns,
        y=pivot_table.index,
        colorscale='YlOrRd',
        text=pivot_table.values.round(2),
        texttemplate='%{text}',
        textfont={"size": 10},
        colorbar=dict(title="Relative Error<br>(Norm. by Rest)")
    ))
    fig.update_layout(
        title='Relative Error Magnitude by Network and Disruption Type',
        xaxis_title='Disruption Type',
        yaxis_title='Network',
        height=800,
        width=900
    )
    return fig

fig_rel_heatmap = plot_relative_error_magnitude_heatmap(region_info)
fig_rel_heatmap.show()

In [ ]:
# =============================================================================
# PHASE 4: TASK-RELEVANCE ANALYSIS
# =============================================================================

print("\n" + "="*80)
print("PHASE 4: TASK-RELEVANCE ANALYSIS")
print("="*80)

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.preprocessing import StandardScaler

# =============================================================================
# 4A. BEHAVIORAL CORRELATION ANALYSIS
# =============================================================================

print("\n" + "="*80)
print("4A. BEHAVIORAL CORRELATION ANALYSIS")
print("="*80)

def calculate_subject_level_metrics(predictions_cv, predictions_task, region_info):
    """
    Calculate per-subject prediction error metrics.
    
    Parameters:
    -----------
    predictions_cv : dict
        Contains 'predictions', 'true_labels', 'subject_ids'
    predictions_task : dict
        Contains 'predictions', 'true_labels', 'subject_ids'
    region_info : DataFrame
        Region-level information with disruption classifications
    
    Returns:
    --------
    DataFrame with subject-level metrics
    """
    # Get unique subjects
    subjects = np.unique(predictions_cv['subject_ids'])
    
    subject_metrics = []
    
    for subj in subjects:
        # Get this subject's predictions
        subj_mask_cv = predictions_cv['subject_ids'] == subj
        subj_mask_task = predictions_task['subject_ids'] == subj
        
        # CV (Rest) performance
        y_true_cv = predictions_cv['true_labels'][subj_mask_cv]
        y_pred_cv = predictions_cv['predictions'][subj_mask_cv]
        acc_cv = (y_true_cv == y_pred_cv).mean()
        
        # Task performance
        y_true_task = predictions_task['true_labels'][subj_mask_task]
        y_pred_task = predictions_task['predictions'][subj_mask_task]
        acc_task = (y_true_task == y_pred_task).mean()
        
        # Calculate per-region errors
        region_errors_cv = []
        region_errors_task = []
        fn_errors = []
        fp_errors = []
        
        for region_idx in region_info.index:
            # Rest errors for this region
            region_mask_cv = y_true_cv == region_idx
            if region_mask_cv.sum() > 0:
                region_acc_cv = (y_pred_cv[region_mask_cv] == region_idx).mean()
                region_errors_cv.append(1 - region_acc_cv)
            
            # Task errors for this region
            region_mask_task = y_true_task == region_idx
            if region_mask_task.sum() > 0:
                region_acc_task = (y_pred_task[region_mask_task] == region_idx).mean()
                region_errors_task.append(1 - region_acc_task)
                
                # Calculate FN and FP for this region
                fn_rate = 1 - (y_pred_task[region_mask_task] == region_idx).mean()
                fp_mask = y_pred_task == region_idx
                if fp_mask.sum() > 0:
                    fp_rate = 1 - (y_true_task[fp_mask] == region_idx).mean()
                else:
                    fp_rate = 0
                
                # Weight by disruption type
                disruption_type = region_info.loc[region_idx, 'Disruption_Type']
                if disruption_type == 'FN-dominated':
                    fn_errors.append(fn_rate)
                elif disruption_type == 'FP-dominated':
                    fp_errors.append(fp_rate)
        
        # Aggregate metrics
        subject_metrics.append({
            'subject_id': subj,
            'accuracy_rest': acc_cv,
            'accuracy_task': acc_task,
            'accuracy_drop': acc_cv - acc_task,
            'avg_error_magnitude_rest': np.mean(region_errors_cv) if region_errors_cv else np.nan,
            'avg_error_magnitude_task': np.mean(region_errors_task) if region_errors_task else np.nan,
            'fn_dominated_error': np.mean(fn_errors) if fn_errors else np.nan,
            'fp_dominated_error': np.mean(fp_errors) if fp_errors else np.nan,
            'n_samples_rest': subj_mask_cv.sum(),
            'n_samples_task': subj_mask_task.sum()
        })
    
    return pd.DataFrame(subject_metrics)

def load_behavioral_data(behavioral_file=None):
    """
    Load behavioral data (accuracy, RT, difficulty).
    
    Parameters:
    -----------
    behavioral_file : str, optional
        Path to behavioral data file (CSV with columns: subject_id, accuracy, rt, difficulty)
    
    Returns:
    --------
    DataFrame with behavioral metrics
    """
    # If no file provided, generate synthetic data for demonstration
    if behavioral_file is None:
        print("⚠ No behavioral file provided. Generating synthetic data for demonstration.")
        
        # Use subject IDs from predictions
        subjects = np.unique(full_ovr_task['subject_ids'])
        
        behavioral_data = pd.DataFrame({
            'subject_id': subjects,
            'task_accuracy': np.random.uniform(0.6, 0.95, len(subjects)),
            'mean_rt': np.random.uniform(400, 800, len(subjects)),
            'task_difficulty': np.random.choice(['easy', 'medium', 'hard'], len(subjects))
        })
        
        return behavioral_data
    else:
        return pd.read_csv(behavioral_file)

# Calculate subject-level metrics
print("\nCalculating subject-level prediction error metrics...")
subject_metrics = calculate_subject_level_metrics(
    full_ovr_cv, full_ovr_task, region_info
)

# Load behavioral data
print("\nLoading behavioral data...")
behavioral_data = load_behavioral_data()  # Replace with actual file path

# Merge datasets
print("\nMerging neural and behavioral data...")
analysis_df = subject_metrics.merge(behavioral_data, on='subject_id', how='inner')

print(f"✓ Combined dataset: {len(analysis_df)} subjects")
print(f"  Columns: {list(analysis_df.columns)}")

# =============================================================================
# Correlation Analysis: Prediction Error vs Behavioral Performance
# =============================================================================

print("\n" + "-"*80)
print("CORRELATION ANALYSIS: Prediction Error vs Task Performance")
print("-"*80)

def correlate_errors_with_behavior(analysis_df):
    """Calculate correlations between prediction errors and behavioral metrics."""
    
    results = []
    
    # Define error metrics and behavioral metrics
    error_metrics = [
        'accuracy_drop',
        'avg_error_magnitude_task',
        'fn_dominated_error',
        'fp_dominated_error'
    ]
    
    behavioral_metrics = ['task_accuracy', 'mean_rt']
    
    for error_metric in error_metrics:
        for behav_metric in behavioral_metrics:
            # Remove NaN values
            valid_mask = ~(analysis_df[error_metric].isna() | analysis_df[behav_metric].isna())
            
            if valid_mask.sum() < 3:
                continue
            
            x = analysis_df.loc[valid_mask, error_metric]
            y = analysis_df.loc[valid_mask, behav_metric]
            
            # Pearson correlation
            r, p = stats.pearsonr(x, y)
            
            # Spearman correlation (robust to outliers)
            rho, p_spearman = stats.spearmanr(x, y)
            
            results.append({
                'Error_Metric': error_metric,
                'Behavioral_Metric': behav_metric,
                'Pearson_r': r,
                'Pearson_p': p,
                'Spearman_rho': rho,
                'Spearman_p': p_spearman,
                'N': valid_mask.sum()
            })
    
    return pd.DataFrame(results)

correlation_results = correlate_errors_with_behavior(analysis_df)

print("\nCorrelation Results:")
print("="*100)
for _, row in correlation_results.iterrows():
    sig_marker = "***" if row['Pearson_p'] < 0.001 else "**" if row['Pearson_p'] < 0.01 else "*" if row['Pearson_p'] < 0.05 else ""
    print(f"{row['Error_Metric']:<30} vs {row['Behavioral_Metric']:<20}")
    print(f"  Pearson r = {row['Pearson_r']:>6.3f} (p = {row['Pearson_p']:.4f}) {sig_marker}")
    print(f"  Spearman ρ = {row['Spearman_rho']:>6.3f} (p = {row['Spearman_p']:.4f})")
    print(f"  N = {row['N']}")
    print()

# =============================================================================
# Hypothesis Testing: Appropriate vs Inappropriate Reorganization
# =============================================================================

print("\n" + "-"*80)
print("HYPOTHESIS TEST: Reorganization Pattern vs Performance")
print("-"*80)

# Define high/low performers
median_accuracy = analysis_df['task_accuracy'].median()
analysis_df['performance_group'] = analysis_df['task_accuracy'].apply(
    lambda x: 'High' if x >= median_accuracy else 'Low'
)

print(f"\nPerformance Groups (median split at {median_accuracy:.3f}):")
print(f"  High performers: {(analysis_df['performance_group'] == 'High').sum()} subjects")
print(f"  Low performers: {(analysis_df['performance_group'] == 'Low').sum()} subjects")

# Compare error patterns
print("\nError Patterns by Performance Group:")
print("-"*80)

for error_metric in ['fn_dominated_error', 'fp_dominated_error', 'accuracy_drop']:
    high_perf = analysis_df[analysis_df['performance_group'] == 'High'][error_metric].dropna()
    low_perf = analysis_df[analysis_df['performance_group'] == 'Low'][error_metric].dropna()
    
    if len(high_perf) > 0 and len(low_perf) > 0:
        t_stat, p_val = stats.ttest_ind(high_perf, low_perf)
        
        print(f"\n{error_metric}:")
        print(f"  High performers: M = {high_perf.mean():.3f}, SD = {high_perf.std():.3f}")
        print(f"  Low performers:  M = {low_perf.mean():.3f}, SD = {low_perf.std():.3f}")
        print(f"  t({len(high_perf) + len(low_perf) - 2}) = {t_stat:.3f}, p = {p_val:.4f}")

# =============================================================================
# Visualization 1: Scatter Plots - Error vs Performance
# =============================================================================

def plot_error_behavior_correlations(analysis_df, correlation_results):
    """Create scatter plots showing error-behavior relationships."""
    
    # Filter for significant correlations
    sig_corrs = correlation_results[correlation_results['Pearson_p'] < 0.05].copy()
    
    if len(sig_corrs) == 0:
        print("⚠ No significant correlations found. Showing strongest correlations instead.")
        sig_corrs = correlation_results.nsmallest(4, 'Pearson_p')
    
    n_plots = len(sig_corrs)
    n_cols = 2
    n_rows = (n_plots + 1) // 2
    
    fig = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=[
            f"{row['Error_Metric']} vs {row['Behavioral_Metric']}<br>r={row['Pearson_r']:.3f}, p={row['Pearson_p']:.3f}"
            for _, row in sig_corrs.iterrows()
        ],
        vertical_spacing=0.12,
        horizontal_spacing=0.1
    )
    
    for idx, (_, row) in enumerate(sig_corrs.iterrows()):
        plot_row = idx // n_cols + 1
        plot_col = idx % n_cols + 1
        
        error_metric = row['Error_Metric']
        behav_metric = row['Behavioral_Metric']
        
        # Remove NaN
        valid_mask = ~(analysis_df[error_metric].isna() | analysis_df[behav_metric].isna())
        plot_df = analysis_df[valid_mask].copy()
        
        # Add scatter
        fig.add_trace(
            go.Scatter(
                x=plot_df[error_metric],
                y=plot_df[behav_metric],
                mode='markers',
                marker=dict(size=8, opacity=0.6, color='steelblue'),
                name=f"{error_metric}",
                showlegend=False,
                hovertemplate=f"<b>Subject %{{customdata}}</b><br>{error_metric}: %{{x:.3f}}<br>{behav_metric}: %{{y:.3f}}<extra></extra>",
                customdata=plot_df['subject_id']
            ),
            row=plot_row, col=plot_col
        )
        
        # Add regression line
        z = np.polyfit(plot_df[error_metric], plot_df[behav_metric], 1)
        p = np.poly1d(z)
        x_line = np.linspace(plot_df[error_metric].min(), plot_df[error_metric].max(), 100)
        
        fig.add_trace(
            go.Scatter(
                x=x_line,
                y=p(x_line),
                mode='lines',
                line=dict(color='red', dash='dash', width=2),
                showlegend=False,
                hoverinfo='skip'
            ),
            row=plot_row, col=plot_col
        )
        
        # Update axes
        fig.update_xaxes(title_text=error_metric.replace('_', ' ').title(), row=plot_row, col=plot_col)
        fig.update_yaxes(title_text=behav_metric.replace('_', ' ').title(), row=plot_row, col=plot_col)
    
    fig.update_layout(
        title="Prediction Error vs Behavioral Performance",
        height=300 * n_rows,
        width=900,
        showlegend=False
    )
    
    return fig

fig_correlations = plot_error_behavior_correlations(analysis_df, correlation_results)
fig_correlations.show()

# =============================================================================
# Visualization 2: Performance Groups Comparison
# =============================================================================

def plot_performance_group_comparison(analysis_df):
    """Compare error patterns between high and low performers."""
    
    error_metrics = ['fn_dominated_error', 'fp_dominated_error', 'accuracy_drop']
    
    fig = go.Figure()
    
    for error_metric in error_metrics:
        high_perf = analysis_df[analysis_df['performance_group'] == 'High'][error_metric].dropna()
        low_perf = analysis_df[analysis_df['performance_group'] == 'Low'][error_metric].dropna()
        
        fig.add_trace(go.Box(
            y=high_perf,
            name=f"{error_metric} (High)",
            marker_color='lightblue',
            boxmean='sd'
        ))
        
        fig.add_trace(go.Box(
            y=low_perf,
            name=f"{error_metric} (Low)",
            marker_color='lightcoral',
            boxmean='sd'
        ))
    
    fig.update_layout(
        title="Error Patterns by Performance Group (High vs Low Task Accuracy)",
        yaxis_title="Error Magnitude",
        xaxis_title="Error Metric and Performance Group",
        height=600,
        width=1000,
        boxmode='group'
    )
    
    return fig

fig_groups = plot_performance_group_comparison(analysis_df)
fig_groups.show()

# =============================================================================
# 4B. TASK-POSITIVE VS TASK-NEGATIVE ANALYSIS
# =============================================================================

print("\n" + "="*80)
print("4B. TASK-POSITIVE VS TASK-NEGATIVE REGIONS")
print("="*80)

# Define task-positive and task-negative networks
TASK_POSITIVE_NETWORKS = [
    'ContA', 'ContB', 'ContC',           # Frontoparietal/Control
    'DorsAttnA', 'DorsAttnB',           # Dorsal Attention
    'SalVentAttnA', 'SalVentAttnB'      # Salience/Ventral Attention
]

TASK_NEGATIVE_NETWORKS = [
    'DefaultA', 'DefaultB', 'DefaultC',  # Default Mode Network
    'LimbicA', 'LimbicB'                 # Limbic (partially task-negative)
]

TASK_SENSORY_NETWORKS = [
    'VisCent', 'VisPeri',               # Visual
    'SomMotA', 'SomMotB'                # Somatomotor
]

# Classify regions
def classify_task_relevance(network):
    """Classify network as task-positive, task-negative, or sensory."""
    if network in TASK_POSITIVE_NETWORKS:
        return 'Task-Positive'
    elif network in TASK_NEGATIVE_NETWORKS:
        return 'Task-Negative'
    elif network in TASK_SENSORY_NETWORKS:
        return 'Sensory'
    else:
        return 'Subcortical'

region_info['Task_Relevance'] = region_info['network'].apply(classify_task_relevance)

print("\nTask Relevance Classification:")
print("="*80)
task_relevance_counts = region_info['Task_Relevance'].value_counts()
for category, count in task_relevance_counts.items():
    pct = (count / len(region_info)) * 100
    print(f"  {category:<20}: {count:>3} regions ({pct:>5.1f}%)")

# =============================================================================
# Compare Disruption Patterns by Task Relevance
# =============================================================================

print("\n" + "-"*80)
print("DISRUPTION PATTERNS BY TASK RELEVANCE")
print("-"*80)

disruption_by_relevance = region_info.groupby(
    ['Task_Relevance', 'Disruption_Type']
).size().unstack(fill_value=0)

print("\nRegion Counts:")
print(disruption_by_relevance)

# Calculate percentages
disruption_by_relevance_pct = disruption_by_relevance.div(
    disruption_by_relevance.sum(axis=1), axis=0
) * 100

print("\nPercentages:")
print(disruption_by_relevance_pct.round(1))

# =============================================================================
# Statistical Test: Task-Positive vs Task-Negative
# =============================================================================

print("\n" + "-"*80)
print("HYPOTHESIS TEST: Task-Negative → High FN, Task-Positive → Stable")
print("-"*80)

# Compare FN rates
task_pos = region_info[region_info['Task_Relevance'] == 'Task-Positive']
task_neg = region_info[region_info['Task_Relevance'] == 'Task-Negative']

print(f"\nFN-dominated regions:")
fn_task_pos = (task_pos['Disruption_Type'] == 'FN-dominated').sum() / len(task_pos) * 100
fn_task_neg = (task_neg['Disruption_Type'] == 'FN-dominated').sum() / len(task_neg) * 100

print(f"  Task-Positive: {fn_task_pos:.1f}%")
print(f"  Task-Negative: {fn_task_neg:.1f}%")

# Chi-square test
from scipy.stats import chi2_contingency

contingency_table = pd.crosstab(
    region_info['Task_Relevance'],
    region_info['Disruption_Type']
)

chi2, p_val, dof, expected = chi2_contingency(contingency_table)
print(f"\nChi-square test: χ²({dof}) = {chi2:.3f}, p = {p_val:.4f}")

# Compare error magnitudes
print("\n" + "-"*80)
print("Error Magnitude by Task Relevance:")
print("-"*80)

for relevance in ['Task-Positive', 'Task-Negative', 'Sensory']:
    regions_subset = region_info[region_info['Task_Relevance'] == relevance]
    
    if len(regions_subset) > 0:
        print(f"\n{relevance}:")
        print(f"  Mean Error Magnitude: {regions_subset['Error_Magnitude'].mean():.2f}")
        print(f"  Mean Recall Gap:      {regions_subset['Recall_Gap'].mean():.2f}pt")
        print(f"  Mean Precision Gap:   {regions_subset['Precision_Gap'].mean():.2f}pt")

# =============================================================================
# Visualization 3: Disruption by Task Relevance
# =============================================================================

def plot_disruption_by_task_relevance(region_info):
    """Stacked bar chart of disruption types by task relevance."""
    
    # Calculate percentages
    relevance_order = ['Task-Positive', 'Sensory', 'Task-Negative', 'Subcortical']
    
    fig = go.Figure()
    
    colors = {
        'FN-dominated': '#E74C3C',
        'FP-dominated': '#F39C12',
        'Both (FN+FP)': '#9B59B6',
        'Stable': '#2ECC71',
        'Improved': '#3498DB'
    }
    
    for disruption_type in ['FN-dominated', 'FP-dominated', 'Both (FN+FP)', 'Stable', 'Improved']:
        if disruption_type not in disruption_by_relevance_pct.columns:
            continue
        
        y_values = [
            disruption_by_relevance_pct.loc[rel, disruption_type] 
            if rel in disruption_by_relevance_pct.index else 0
            for rel in relevance_order
        ]
        
        fig.add_trace(go.Bar(
            name=disruption_type,
            x=relevance_order,
            y=y_values,
            marker_color=colors.get(disruption_type, 'gray'),
            text=[f"{v:.1f}%" for v in y_values],
            textposition='inside'
        ))
    
    fig.update_layout(
        barmode='stack',
        title="Disruption Patterns by Task Relevance<br><sub>Expected: Task-Negative → High FN (suppressed), Task-Positive → Stable</sub>",
        xaxis_title='Task Relevance Category',
        yaxis_title='Percentage of Regions (%)',
        height=600,
        width=900,
        legend_title='Disruption Type'
    )
    
    return fig

fig_task_relevance = plot_disruption_by_task_relevance(region_info)
fig_task_relevance.show()

# =============================================================================
# Visualization 4: Error Magnitude Heatmap by Task Relevance
# =============================================================================

def plot_task_relevance_heatmap(region_info):
    """Heatmap of average gaps by task relevance and disruption type."""
    
    # Calculate average gaps
    pivot_recall = region_info.groupby(['Task_Relevance', 'Disruption_Type'])['Recall_Gap'].mean().reset_index()
    pivot_precision = region_info.groupby(['Task_Relevance', 'Disruption_Type'])['Precision_Gap'].mean().reset_index()
    
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Average Recall Gap', 'Average Precision Gap'),
        horizontal_spacing=0.15
    )
    
    for idx, (data, title) in enumerate([(pivot_recall, 'Recall_Gap'), (pivot_precision, 'Precision_Gap')]):
        pivot_table = data.pivot(index='Task_Relevance', columns='Disruption_Type', values=title)
        
        fig.add_trace(
            go.Heatmap(
                z=pivot_table.values,
                x=pivot_table.columns,
                y=pivot_table.index,
                colorscale='RdYlBu_r',
                text=pivot_table.values.round(2),
                texttemplate='%{text}',
                textfont={"size": 10},
                showscale=(idx == 1),
                colorbar=dict(title="Gap (pt)") if idx == 1 else None
            ),
            row=1, col=idx+1
        )
    
    fig.update_layout(
        title='Performance Gaps by Task Relevance and Disruption Type',
        height=500,
        width=1200
    )
    
    return fig

fig_heatmap_relevance = plot_task_relevance_heatmap(region_info)
fig_heatmap_relevance.show()

# =============================================================================
# Save Results
# =============================================================================

# Add task relevance to region info
region_info.to_csv('region_info_with_task_relevance.csv', index=False)

# Save subject-level analysis
analysis_df.to_csv('subject_behavioral_analysis.csv', index=False)

# Save correlation results
correlation_results.to_csv('error_behavior_correlations.csv', index=False)

print("\n" + "="*80)
print("✓ PHASE 4 COMPLETE")
print("="*80)
print("\nResults saved to:")
print("  - region_info_with_task_relevance.csv")
print("  - subject_behavioral_analysis.csv")
print("  - error_behavior_correlations.csv")

print("\n" + "="*80)
print("KEY FINDINGS SUMMARY")
print("="*80)
print("\n1. Behavioral Correlations:")
print(f"   - Strongest correlation: {correlation_results.nsmallest(1, 'Pearson_p').iloc[0]['Error_Metric']}")
print(f"     vs {correlation_results.nsmallest(1, 'Pearson_p').iloc[0]['Behavioral_Metric']}")
print(f"     r = {correlation_results.nsmallest(1, 'Pearson_p').iloc[0]['Pearson_r']:.3f}")

print("\n2. Task Relevance Patterns:")
print(f"   - Task-Positive FN rate: {fn_task_pos:.1f}%")
print(f"   - Task-Negative FN rate: {fn_task_neg:.1f}%")
print(f"   - Hypothesis supported: {'YES' if fn_task_neg > fn_task_pos else 'NO'}")


PHASE 4: TASK-RELEVANCE ANALYSIS

4A. BEHAVIORAL CORRELATION ANALYSIS

Calculating subject-level prediction error metrics...


KeyError: 'subject_ids'